# Train QA Generation Model — Llama-3.2-3B + QLoRA on SQuAD v2

**Environment:** Kaggle T4x2 (2x NVIDIA Tesla T4, 16GB each)

**Task:** Fine-tune Llama to generate (Question, Answer) from context passages

**Dataset:** SQuAD v2 (filtered to answerable questions only)

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [2]:
import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Core packages
install("unsloth")
install("trl>=0.8.6")
install("peft>=0.10.0")
install("bitsandbytes>=0.43.0")
install("accelerate>=0.27.0")
install("datasets>=2.14.0")
install("transformers>=4.40.0")
install("sentencepiece")
install("protobuf")

print("All packages installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

All packages installed.


In [3]:
import os
import json
import random
import torch
from kaggle_secrets import UserSecretsClient

# Get HuggingFace token from Kaggle Secrets
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN not found. Add it to Kaggle Secrets.")

# Optional: W&B (comment out if not needed)
# WANDB_KEY = secrets.get_secret("WANDB_API_KEY")
# import wandb
# wandb.login(key=WANDB_KEY)

# Print GPU info
print(f"Available GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory / 1024**3:.1f} GB")

# Configuration
CONFIG = {
    "base_model": "meta-llama/Llama-3.2-3B-Instruct",
    "output_dir": "/kaggle/working/qa_model_output",
    "hf_repo": "",          # Set to "username/llama-qa-squad" to push to Hub
    "max_train_samples": 25000,
    "max_val_samples": 1000,
    "max_seq_length": 512,
    # Training hyperparameters
    "num_epochs": 2,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 8,
    "gradient_accumulation_steps": 1,  # Effective batch = 2*2*8 = 32
    "learning_rate": 2e-4,
    "weight_decay": 0.001,
    "warmup_ratio": 0.05,
    "lr_scheduler_type": "cosine",
    "save_steps": 500,
    "eval_steps": 500,
    "logging_steps": 100,
    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
}

HF_TOKEN loaded from Kaggle Secrets.
Available GPUs: 2
  GPU 0: Tesla T4, 14.6 GB
  GPU 1: Tesla T4, 14.6 GB


In [4]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["base_model"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,           # Auto-detect
    load_in_4bit=True,
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
USING_UNSLOTH = True
print("Using Unsloth for accelerated training.")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # For training (left for inference)

model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Using Unsloth for accelerated training.
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [5]:
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are an expert educator. Given a passage, generate one clear "
    "question and its correct answer."
)

def build_qa_training_text(sample: dict) -> dict:
    """Convert SQuAD sample to training text using Llama chat template."""

    context = sample["context"].strip()
    question = sample["question"].strip()
    answers = sample["answers"]["text"]

    if len(answers) == 0:
        return {"text": "", "valid": False}

    answer = answers[0].strip()

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": (
                f"Passage:\n{context}\n\n"
                "Generate ONE question together with its correct answer "
                "based only on the passage."
            ),
        },
        {
            "role": "assistant",
            "content": (
                f"Question: {question}\n"
                f"Answer: {answer}"
            ),
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {
        "text": text,
        "valid": True,
    }

print("Loading SQuAD v2...")
train_raw = load_dataset("squad_v2", split="train")
val_raw = load_dataset("squad_v2", split="validation")

# Filter: keep only answerable questions
train_raw = train_raw.filter(lambda x: len(x["answers"]["text"]) > 0)
val_raw = val_raw.filter(lambda x: len(x["answers"]["text"]) > 0)
print(f"Train: {len(train_raw):,} answerable samples")
print(f"Val:   {len(val_raw):,} answerable samples")

# Subsample
if CONFIG["max_train_samples"] and len(train_raw) > CONFIG["max_train_samples"]:
    train_raw = train_raw.shuffle(seed=42).select(range(CONFIG["max_train_samples"]))
if CONFIG["max_val_samples"] and len(val_raw) > CONFIG["max_val_samples"]:
    val_raw = val_raw.shuffle(seed=42).select(range(CONFIG["max_val_samples"]))

# Format
train_dataset = train_raw.map(
    build_qa_training_text,
    remove_columns=train_raw.column_names,
    num_proc=4,
    desc="Formatting train"
)
val_dataset = val_raw.map(
    build_qa_training_text,
    remove_columns=val_raw.column_names,
    num_proc=4,
    desc="Formatting val"
)

# Keep only valid samples
train_dataset = train_dataset.filter(lambda x: x["valid"])
val_dataset = val_dataset.filter(lambda x: x["valid"])
train_dataset = train_dataset.remove_columns(["valid"])
val_dataset = val_dataset.remove_columns(["valid"])

print(f"Final train: {len(train_dataset):,}")
print(f"Final val:   {len(val_dataset):,}")

Loading SQuAD v2...


README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

...

Formatting train (num_proc=4):   0%|          | 0/25000 [00:00<?, ? examples/s]

Formatting val (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Final train: 25,000
Final val:   1,000


In [6]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    bf16=False,
    fp16=True,
    optim="paged_adamw_32bit",
    gradient_checkpointing="unsloth",
    eval_strategy="steps",      
    eval_steps=500,             
    save_strategy="steps",      
    save_steps=500,              
    logging_steps=CONFIG["logging_steps"],
    report_to="none",
    save_total_limit=1,          
    load_best_model_at_end=True, 
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    dataloader_num_workers=2,    
    dataloader_pin_memory=True,  
    dataset_num_proc=4,          
    ddp_find_unused_parameters=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    # formatting_func=lambda x: tokenizer.apply_chat_template(
    #     x["messages"],
    #     tokenize=False,
    #     add_generation_prompt=False,
    # ),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/25000 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=4):   0%|          | 0/25000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [7]:
import time

print("Starting training...")
start_time = time.time()

trainer_stats = trainer.train()

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed / 3600:.2f} hours.")
print(f"Train loss: {trainer_stats.training_loss:.4f}")

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,990 | Num Epochs = 2 | Total steps = 1,624
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
500,1.567228,1.642374
1000,1.343594,1.688177
1500,1.309601,1.701452
1624,1.299529,1.702139


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qa_model_output/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qa_model_output/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qa_model_output/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qa_model_output/checkpoint-1624/tokenizer_config.json.



Training completed in 5.02 hours.
Train loss: 1.4677


In [8]:
from huggingface_hub import login

# Save locally
os.makedirs(CONFIG["output_dir"], exist_ok=True)
model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"Model saved to: {CONFIG['output_dir']}")

# Also save merged model (optional — larger but faster for inference)
# Comment this out if storage is limited
MERGED_DIR = CONFIG["output_dir"] + "_merged"
try:
    if USING_UNSLOTH:
        model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
    else:
        merged = model.merge_and_unload()
        merged.save_pretrained(MERGED_DIR)
        tokenizer.save_pretrained(MERGED_DIR)
    print(f"Merged model saved to: {MERGED_DIR}")
except Exception as e:
    print(f"Warning: Could not save merged model: {e}")

# Push adapter to HuggingFace Hub
if CONFIG["hf_repo"]:
    login(token=HF_TOKEN)
    model.push_to_hub(CONFIG["hf_repo"])
    tokenizer.push_to_hub(CONFIG["hf_repo"])
    print(f"Pushed adapter to: https://huggingface.co/{CONFIG['hf_repo']}")
else:
    print("HF_REPO not set — skipping Hub push.")
    print("Download the model from Kaggle output: /kaggle/working/qa_model_output")

# List output files
import os
for root, dirs, files in os.walk(CONFIG["output_dir"]):
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path) / 1024 / 1024
        print(f"  {path} ({size:.1f} MB)")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qa_model_output/tokenizer_config.json.


Model saved to: /kaggle/working/qa_model_output


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qa_model_output_merged/tokenizer_config.json.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:17<00:00,  8.50s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:48<00:00, 24.15s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/qa_model_output_merged`
Merged model saved to: /kaggle/working/qa_model_output_merged
HF_REPO not set — skipping Hub push.
Download the model from Kaggle output: /kaggle/working/qa_model_output
  /kaggle/working/qa_model_output/adapter_model.safetensors (92.8 MB)
  /kaggle/working/qa_model_output/README.md (0.0 MB)
  /kaggle/working/qa_model_output/tokenizer.json (16.4 MB)
  /kaggle/working/qa_model_output/chat_template.jinja (0.0 MB)
  /kaggle/working/qa_model_output/tokenizer_config.json (0.0 MB)
  /kaggle/working/qa_model_output/adapter_config.json (0.0 MB)
  /kaggle/working/qa_model_output/checkpoint-500/trainer_state.json (0.0 MB)
  /kaggle/working/qa_model_output/checkpoint-500/adapter_model.safetensors (92.8 MB)
  /kaggle/working/qa_model_output/checkpoint-500/README.md (0.0 MB)
  /kaggle/working/qa_model_output/checkpoint-500/scaler.pt (0.0 MB)
  /kaggle/working/qa_model_output/checkpoint-500/tokenizer.json (16.4 MB)
  